In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
!pip install gpboost
import gpboost as gpb
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()

df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# binary label for classification metrics
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y = df["phq8_score"].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

gkf = GroupKFold(n_splits=5)

results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    y_bin_test = y_bin[test_idx]

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # convert regression output → binary prediction
    preds_bin = (preds >= 10).astype(int)

    results.append({
        "fold": fold,

        # regression metrics
        "mae": mean_absolute_error(y_test, preds),
        "rmse": mean_squared_error(y_test, preds) ** 0.5,
        "r2": r2_score(y_test, preds),

        # classification metrics (for comparison with Androids)
        "accuracy": accuracy_score(y_bin_test, preds_bin),
        "f1": f1_score(y_bin_test, preds_bin),
        "roc_auc": roc_auc_score(y_bin_test, preds)
    })

results_df = pd.DataFrame(results)

print(results_df)

print("\nMean results:")
print(results_df.mean(numeric_only=True))

# Save results
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")

results_df.to_csv(RESULTS_PATH / "Radar_xgb.csv", index=False)

   fold       mae      rmse        r2  accuracy        f1   roc_auc
0     1  5.679108  6.678884 -0.035276  0.568409  0.354697  0.534175
1     2  4.922508  6.130742 -0.055729  0.530241  0.346405  0.498402
2     3  5.171609  6.257674 -0.043341  0.554903  0.343154  0.528159
3     4  4.496938  5.408797 -0.013323  0.600117  0.334311  0.550000
4     5  4.938764  6.050746 -0.022011  0.593658  0.379928  0.551022

Mean results:
fold        3.000000
mae         5.041785
rmse        6.105369
r2         -0.033936
accuracy    0.569466
f1          0.351699
roc_auc     0.532352
dtype: float64


In [21]:
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()

df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# Binary target
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

gkf = GroupKFold(n_splits=5)

results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_bin, groups=groups), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_bin[train_idx], y_bin[test_idx]

    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results)

print(results_df)
print("\nMean results:")
print(results_df.mean(numeric_only=True))

RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

results_df.to_csv(RESULTS_PATH / "radar_xgbc.csv", index=False)

   fold  accuracy        f1   roc_auc
0     1  0.549618  0.293088  0.498692
1     2  0.533764  0.353420  0.482620
2     3  0.568409  0.351280  0.536442
3     4  0.593658  0.326848  0.537983
4     5  0.594245  0.351174  0.549367

Mean results:
fold        3.000000
accuracy    0.567939
f1          0.335162
roc_auc     0.521021
dtype: float64


GPBoost accounts for group random effects by selecting a source of data clustering.

In this case, ID of the participants served as the source of clustering.

In [ ]:
# Paths
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_csv(dataset)

# Feature columns
feature_cols = df.iloc[:, 12:-1].columns.tolist()

# Clean
df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()
df["depressed"] = (df["phq8_score"] >= 10).astype(int)
df["group_code"] = pd.factorize(df["participant_id"])[0].astype(np.int32)

# Arrays
X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["group_code"].values

# CV
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    group_train = groups[train_idx].astype(np.int32) # group codes for training data
    group_test = groups[test_idx].astype(np.int32)

    gp_model = gpb.GPModel(group_data=group_train) # GP model for group random effects
    train_data = gpb.Dataset(X_train, label=y_train)

    params = {
        "objective": "binary",
        "learning_rate": 0.05,
        "max_depth": 4,
        "verbose": 0
    }

    gpbst = gpb.train(
        params=params,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=200
    )

    pred = gpbst.predict(
        data=X_test,
        group_data_pred=group_test,
        predict_var=False
    )

    p_gpboost = pred["response_mean"]
    y_pred = (p_gpboost >= 0.5).astype(int)

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, p_gpboost),
        "logloss": log_loss(y_test, p_gpboost)
    })

# Fold-level results
fold_results_df = pd.DataFrame(fold_results)
print("Fold-level results:")
print(fold_results_df)

# Overall summary
summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df),
    "n_depressed": int(df["depressed"].sum()),
    "n_control": int((df["depressed"] == 0).sum()),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std(),
    "logloss_mean": fold_results_df["logloss"].mean(),
    "logloss_std": fold_results_df["logloss"].std(),
}])

print("\nOverall average metrics:")
print(summary_df)

# Save
fold_results_df.to_csv(RESULTS_PATH / "radar_gpboost_folds.csv", index=False)
summary_df.to_csv(RESULTS_PATH / "radar_gpboost_summary.csv", index=False)

[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
[GPBoost] [Warning] The 'objective' (='binary') f